# 사고 빈도 모델링 1~4단계 (statsmodels Poisson 회귀)

| 단계 | 하는 일 | 전처리는 어디서? |
|---|---|---|
| 0 | 데이터 불러오기 + **고정 규칙** 전처리 (cap, Area 제거, 나이 구간, log) | 데이터를 보고 배우는 게 없음 → 분할 **전**에 해도 누수 없음 |
| 1 | 스케일링할 피처 확인 + 전처리 **설계** (Other 합치기, 표준화, 원-핫) | 여기서는 **정의만** 하고 fit하지 않음 |
| 2 | X, Y를 새 DataFrame으로 추출 | |
| 3 | train_test_split으로 분리 (같은 계약은 같은 쪽으로) | |
| 4 | fit() · predict() (statsmodels Poisson 회귀, offset = log 가입 기간) | 1단계에서 설계한 전처리가 **train으로만** fit됨 |

statsmodels 모델을 sklearn 모양(`fit` · `predict`)으로 감싸서 `Pipeline`에 넣음
→ 5단계 교차 검증·6단계 GridSearchCV를 sklearn 그대로 쓰면서, 겹마다 전처리를 자동으로 다시 배움 (누수 없음)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_poisson_deviance, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# ================================================================ 0단계: 불러오기 + 고정 규칙 전처리
# 고정된 숫자(4, 1, 20, 150, 12, 나이 경계)를 행마다 적용할 뿐 → 분할 전에 해도 누수 없음
freq = pd.read_csv("freMTPL2freq.csv")
print(f"계약 수: {len(freq):,} · 사고 {freq.ClaimNb.sum():,}건 · "
      f"1년당 사고율 {freq.ClaimNb.sum() / freq.Exposure.sum():.4f}")

# ---------------------------------------------------------------- 묶음 번호 (3단계 분할용)
# 한 계약이 여러 IDpol로 쪼개져 있음 → 특성이 모두 같은 행 = 같은 묶음
# 자르기(clip) 전 원본 값으로 만들어야 서로 다른 계약이 덜 섞임
FEAT = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand",
        "VehGas", "Density", "Region"]
freq["GroupID"] = freq.groupby(FEAT, sort=False).ngroup()
#  .ngroup으로 번호 부여

# ---------------------------------------------------------------- 타깃 · 가입 기간
# Y = ClaimNb(사고 건수), 가입 기간은 4단계에서 offset = log(Exposure)로 씀
# 사고 5건 이상 9건은 기록 오류로 보임 → 지우지 않고 4로 cap ("위험하다"는 정보는 남김)
freq["ClaimNb"] = freq["ClaimNb"].clip(upper=4)
# 1년 넘는 계약은 기록 오차 → 1로 cap (분모가 커지면 사고율이 작게 계산됨)
freq["Exposure"] = freq["Exposure"].clip(upper=1)

# ---------------------------------------------------------------- 피처
# Area는 Density를 6구간으로 자른 것 → 같은 정보가 두 번 들어가므로 제거
freq = freq.drop(columns=["Area"])
AGE_BINS = [17, 21, 25, 30, 40, 50, 60, 70, 80, 101]
freq["VehAge"] = freq["VehAge"].clip(upper=20)
freq["BonusMalus"] = freq["BonusMalus"].clip(upper=150)
freq["DrivAgeBin"] = pd.cut(freq["DrivAge"], bins=AGE_BINS).astype(str)
freq["VehPower"] = freq["VehPower"].clip(upper=12).astype(str)   # 등급별로 따로 보려고 범주로
freq["logDensity"] = np.log(freq["Density"])

# Exposure = 0이면 offset log(0) = -inf → numpy는 경고만 내고 계속 계산함
assert freq["Exposure"].between(0, 1, inclusive="right").all(), "Exposure가 0 이하 또는 1 초과"
assert "nan" not in set(freq["DrivAgeBin"]), "나이 구간에 안 들어간 운전자 있음"
print("[0] 고정 규칙 전처리 완료")

계약 수: 678,013 · 사고 36,102건 · 1년당 사고율 0.1007
[0] 고정 규칙 전처리 완료


In [ ]:
# ================================================================ 1단계: 스케일링할 피처 확인
# 숫자 피처의 범위·치우침(skew)을 보고 정규화(MinMax)와 표준화(Standard) 중 고름
check = ["VehAge", "BonusMalus", "Density", "logDensity", "Exposure"]
desc = freq[check].describe().T[["min", "max", "mean", "std"]]
desc["skew"] = freq[check].skew()
print("[1] 숫자 피처 확인")
print(desc.round(2).to_string())

[1] 숫자 피처 확인
             min      max     mean      std  skew
VehAge       0.0     20.0     6.98     5.40  0.52
BonusMalus  50.0    150.0    59.76    15.61  1.70
Density      1.0  27000.0  1792.42  3958.65  4.65
logDensity   0.0     10.2     5.98     1.87  0.03
Exposure     0.0      1.0     0.53     0.36  0.08


**확인 결과 → 결정**

*   항목 추가
*   항목 추가



| 피처 | 처리 | 이유 |
|---|---|---|
| VehAge, BonusMalus, logDensity | **표준화 (StandardScaler)** | 단위가 제각각(년·점·log 인구)이라 크기를 맞춤 → 계수를 "1 표준편차당 사고율 배수"로 같은 잣대에서 비교. 6단계 규제(alpha)가 모든 계수에 공평하게 걸리려면 필수. BonusMalus처럼 한쪽으로 몰린 값은 MinMax로 하면 대부분이 0 근처로 눌림 |
| Density | log 변환 후 표준화 (원본은 안 씀) | 1~27,000으로 꼬리가 아주 길어서(skew 큼) 그대로 표준화하면 소수 대도시가 계수를 좌우함 |
| Exposure | 스케일링 안 함 | 피처가 아니라 **offset**(log 가입 기간, 계수 1로 고정). 단위가 '년'이어야 의미가 있음 |
| ClaimNb (Y) | 스케일링 안 함 | 타깃. Poisson 회귀는 0, 1, 2…로 세는 값을 그대로 받아야 함 |
| 범주형 (나이 구간, VehPower, VehGas, VehBrand, Region) | 원-핫 인코딩만 | 0/1 값이라 스케일링 의미 없음 |

In [ ]:
# ---------------------------------------------------------------- 1단계: 데이터에서 "배우는" 전처리 + 모델 설계
# 아래는 정의만 함 (fit 안 함) → 4단계 fit()에서 train으로만 배움
NUM_COLS = ["VehAge", "BonusMalus", "logDensity"]
# 연속형 피쳐
CAT_COLS = ["DrivAgeBin", "VehPower", "VehGas", "VehBrand", "Region"]
# 범주형 피쳐


class RareLevelMerger(BaseEstimator, TransformerMixin):
    """사고 수가 min_claims 미만인 범주를 Other로 합침
    fit: 어떤 범주를 남길지 배움 (train에서만) / transform: 배운 목록으로 바꿈 (어디에나)"""
    #BaseEstimator는 get_params와 set_params(GridSearchCV가 쓰는 기능)를 줌.
    #TransformerMixin은 fit_transform(fit과 transform을 한 번에)을 줌.

    def __init__(self, cols=("Region", "VehBrand"), min_claims=200):
        self.cols = cols
        self.min_claims = min_claims
        # 기본 설정값

    def fit(self, X, y):
        # 희귀 범주는 행 수가 아니라 "사고 수(Y)"로 판단
        # (사고 45건인 지역은 5건만 더 나도 사고율이 11% 흔들림)
        claims = pd.Series(np.asarray(y), index=X.index)
        self.keep_ = {}
        # 200이상인 지역이나 브랜드 남기기
        for c in self.cols:
            total = claims.groupby(X[c]).sum()
            # 같은 지역마다 group하고 사고 건수 합하기
            self.keep_[c] = set(total[total >= self.min_claims].index)
            # 200 이상인 지역이나 브랜드 이름 남기기
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            X[c] = X[c].where(X[c].isin(self.keep_[c]), "Other")
            # 남은 지역이나 브랜드(self.keep_[c])에 없는 이름이라면 Other이라 적기
        return X


class SMPoissonGLM(BaseEstimator, RegressorMixin):
    """statsmodels Poisson 회귀를 sklearn 모양(fit · predict)으로 감싼 것
    X의 Exposure 열 → offset = log(Exposure), 나머지 열 → 피처
    alpha = 0이면 일반 Poisson 회귀(summary로 p값 확인 가능), alpha > 0이면 L2(릿지) 규제"""

    def __init__(self, alpha=0.0):
        self.alpha = alpha
        # 기본 설정 람다 값(릿지회귀 대비용)

    def _split(self, X):
        X = pd.DataFrame(X).copy()
        exposure = X.pop("Exposure") if "Exposure" in X else X.pop(X.columns[-1])
        return sm.add_constant(X, has_constant="add"), np.log(np.asarray(exposure, dtype=float))
        # constant = 1인 걸로 다 채운 X 반환 / exposure에 log 취한 값 반환

    def fit(self, X, y):
        exog, offset = self._split(X)
        # 위에 했던 나누기 나눠야 하는 이유는 아래 모형 함수는 exposure까지 포함해서 결과를 내기 때문
        # offset를 따로 둬야하기 때문
        model = sm.GLM(np.asarray(y, dtype=float), exog, family=sm.families.Poisson(), offset=offset)
        if self.alpha == 0:
            self.result_ = model.fit()
        else:
            penalty = np.r_[0.0, np.full(exog.shape[1] - 1, self.alpha)]   # 절편은 규제하지 않음
            self.result_ = model.fit_regularized(alpha=penalty, L1_wt=0)   # L1_wt=0 → L2(릿지)
        self.params_ = np.asarray(self.result_.params)
        return self

    def predict(self, X):
        # 예상 사고 건수 = exp(계수 합 + log 가입 기간) = 가입 기간 × 1년당 사고율
        exog, offset = self._split(X)
        return np.exp(exog.to_numpy(dtype=float) @ self.params_ + offset)
        # 각 계수 반환


def make_model(estimator):
    # 희귀 범주 합치기 → 숫자는 표준화 · 범주는 원-핫 · Exposure는 그대로 통과 → 모델
    encode = ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        # 표준화
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT_COLS),
        # 범주 변수화
        ("offset", "passthrough", ["Exposure"]),      # 스케일링 안 하고 모델에 넘김 (offset용)
    ], verbose_feature_names_out=False).set_output(transform="pandas")   # 열 이름 유지 → summary에 표시
    return Pipeline([("rare", RareLevelMerger()), ("encode", encode), ("model", estimator)])


print("[1] 스케일링 대상:", NUM_COLS)
print("    원-핫 대상:", CAT_COLS)
print("    offset: log(Exposure)")

[1] 스케일링 대상: ['VehAge', 'BonusMalus', 'logDensity']
    원-핫 대상: ['DrivAgeBin', 'VehPower', 'VehGas', 'VehBrand', 'Region']
    offset: log(Exposure)


In [ ]:
# ================================================================ 2단계: X, Y를 새 DataFrame으로 추출
# X = 피처 8개 + Exposure(offset용), Y = 사고 건수
X = freq[["Exposure"] + NUM_COLS + CAT_COLS].copy()
y = freq["ClaimNb"].copy()
groups = freq["GroupID"].copy()
print(f"[2] X {X.shape[0]:,}행 × {X.shape[1]}열 · Y = ClaimNb")
print(X.head(3).to_string())

[2] X 678,013행 × 9열 · Y = ClaimNb
   Exposure  VehAge  BonusMalus  logDensity DrivAgeBin VehPower   VehGas VehBrand       Region
0      0.10       0          50    7.104144   (50, 60]        5  Regular      B12  Rhone-Alpes
1      0.77       0          50    7.104144   (50, 60]        5  Regular      B12  Rhone-Alpes
2      0.75       2          50    3.988984   (50, 60]        6   Diesel      B12     Picardie


In [ ]:
# ================================================================ 3단계: train_test_split으로 분리
# 행이 아니라 "묶음 번호"를 train_test_split으로 나눔 → 같은 계약이 train과 test에 동시에 들어가지 않음
# 묶음에 사고가 있었는지(0/1)로 층화 → 사고 난 계약이 양쪽에 고르게
group_claim = freq.groupby("GroupID")["ClaimNb"].max().clip(upper=1)
g_train, g_test = train_test_split(group_claim.index, test_size=0.2, random_state=42,
                                   stratify=group_claim)
is_train = groups.isin(g_train)

X_train, X_test = X[is_train].copy(), X[~is_train].copy()
y_train, y_test = y[is_train].copy(), y[~is_train].copy()
groups_train = groups[is_train]          # 5단계 교차 검증도 묶음 단위로 (GroupKFold)

assert set(groups[is_train]).isdisjoint(groups[~is_train]), "같은 묶음이 train과 test에 모두 있음"
print(f"[3] train {len(X_train):,} / test {len(X_test):,} (test 비율 {len(X_test) / len(X):.1%})")
print(f"    1년당 사고율 train {y_train.sum() / X_train.Exposure.sum():.4f}"
      f" / test {y_test.sum() / X_test.Exposure.sum():.4f}")

[3] train 542,576 / test 135,437 (test 비율 20.0%)
    1년당 사고율 train 0.1005 / test 0.1009


In [ ]:
# ================================================================ 4단계: fit() · predict() — statsmodels Poisson 회귀
# log(예상 사고 수) = log(가입 기간) + β0 + β1·VehAge + …  → 예측은 항상 0보다 큼
# fit() 한 번에: 희귀 범주 목록·평균·표준편차를 train으로 배우고 → 모델 학습
# alpha = 0: 규제 없는 일반 Poisson 회귀 (6단계에서 alpha를 탐색)
pois = make_model(SMPoissonGLM(alpha=0))
pois.fit(X_train, y_train)

pred_train = pois.predict(X_train)      # 예상 사고 건수
pred_test = pois.predict(X_test)


def evaluate(y_true, pred):
    # Poisson deviance = Poisson 회귀가 최소화하는 기준 → 이 모델의 주 지표 (작을수록 좋음)
    return {"Poisson deviance": mean_poisson_deviance(y_true, pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
            "R²": r2_score(y_true, pred)}


result = pd.DataFrame({"train": evaluate(y_train, pred_train),
                       "test": evaluate(y_test, pred_test)}).T
print("[4] Poisson 회귀 결과")
print(result.round(4).to_string())
print(f"    예상 사고 건수 합: test {pred_test.sum():,.0f}건 / 실제 {y_test.sum():,}건")

[4] Poisson 회귀 결과
       Poisson deviance    RMSE      R²
train            0.3181  0.2364  0.0111
test             0.3209  0.2379  0.0093
    예상 사고 건수 합: test 7,222건 / 실제 7,246건


In [ ]:
# ---------------------------------------------------------------- 4단계: 계수 확인 (statsmodels의 장점)
# summary: 계수 · 표준오차 · p값 · 95% 신뢰구간
res = pois.named_steps["model"].result_
print(res.summary())

# 표준화한 피처: exp(계수) = "1 표준편차 늘 때 사고율 몇 배"
ci = np.exp(res.conf_int().loc[NUM_COLS])
print("\n[4] 1 표준편차 늘 때 사고율 배수 (95% 신뢰구간)")
print(pd.DataFrame({"배수": np.exp(res.params[NUM_COLS]), "하한": ci[0], "상한": ci[1]}).round(3).to_string())

# 7단계 PPT용: train에서 배운 전처리 기준
rare = pois.named_steps["rare"]
scaler = pois.named_steps["encode"].named_transformers_["num"]
print("\n    Other로 합친 Region:", sorted(set(X_train.Region) - rare.keep_["Region"]))
print("    Other로 합친 VehBrand:", sorted(set(X_train.VehBrand) - rare.keep_["VehBrand"]))
print(pd.DataFrame({"평균": scaler.mean_, "표준편차": scaler.scale_}, index=NUM_COLS).round(3).to_string())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:               542576
Model:                            GLM   Df Residuals:                   542528
Model Family:                 Poisson   Df Model:                           47
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1400e+05
Date:                Tue, 27 Oct 2026   Deviance:                   1.7260e+05
Time:                        17:46:25   Pearson chi2:                 1.38e+06
No. Iterations:                     7   Pseudo R-squ. (CS):            0.01156
Covariance Type:            nonrobust                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
cons

In [ ]:
# ---------------------------------------------------------------- 참고: 변수 없는 모델과 비교
# 모두에게 train의 1년당 사고율을 주는 모델 (예상 사고 수 = 평균 사고율 × 가입 기간)
# → Poisson 회귀가 이보다 나아야 피처가 쓸모 있음
# (R²가 작게 나와도 정상: 사고는 우연이 커서 개인 단위로는 원래 잘 안 맞음)
base_rate = y_train.sum() / X_train["Exposure"].sum()
pred_base = base_rate * X_test["Exposure"]
compare = pd.DataFrame({"변수 없음": evaluate(y_test, pred_base),
                        "Poisson 회귀": evaluate(y_test, pred_test)}).T
print("[참고] test 비교")
print(compare.round(4).to_string())
gain = 1 - compare.loc["Poisson 회귀", "Poisson deviance"] / compare.loc["변수 없음", "Poisson deviance"]
print(f"    Poisson deviance가 변수 없는 모델보다 {gain:.1%} 작음")

[참고] test 비교
            Poisson deviance    RMSE      R²
변수 없음                 0.3324  0.2393 -0.0028
Poisson 회귀            0.3209  0.2379  0.0093
    Poisson deviance가 변수 없는 모델보다 3.5% 작음


## 5~6단계: Tweedie 회귀 + 교차 검증 + GridSearchCV

**왜 Tweedie인가 — 과대산포 보완**
- 포아송은 "분산 = 평균"을 가정함. 4단계 summary의 Pearson chi2 / Df Residuals ≈ 2.5 → 실제 분산이 가정보다 약 2.5배 큼(과대산포)
- 과대산포가 있으면 예측값(평균)은 크게 안 틀리지만, 계수의 표준오차·p값이 실제보다 작게(과하게 확신) 나옴
- Tweedie 회귀는 분산 = φ × 평균^power 로 가정 → **power가 하이퍼 파라미터**
  - power = 1 → 포아송과 같음 / 1 < power < 2 → 평균이 클수록 분산이 더 빠르게 커짐(과대산포 반영)
- scikit-learn의 `TweedieRegressor`라서 교재에서 배운 `cross_validate`, `GridSearchCV`를 그대로 쓸 수 있음

**offset 대신 쓰는 방법**
- TweedieRegressor는 offset을 받지 못함 → 타깃 = 1년당 사고율(ClaimNb / Exposure), 가중치(sample_weight) = Exposure
- 가입 기간이 긴 계약은 더 믿을 만한 관측이라 무게를 더 줌. 포아송에서는 offset과 수학적으로 같은 모델이 됨

| 단계 | 셀 | 하는 일 |
|---|---|---|
| 5·6 준비 | 아래 첫 번째 셀 | Tweedie 파이프라인, 평가 기준(사고 건수 Poisson deviance), GroupKFold 정의 |
| 5 | 두 번째 셀 | 포아송(power=1)을 GroupKFold 5겹 교차 검증으로 평가 → 기준 점수 + 과대산포 확인 |
| 6 | 세 번째 셀 | power × alpha 조합을 GridSearchCV로 탐색 → 교차 검증 점수가 가장 좋은 조합 선택 |
| 6 마무리 | 네 번째 셀 | 고른 조합으로 train 전체 재학습 → test로 **딱 한 번** 최종 평가 |

In [ ]:
# ================================================================ 5·6단계 준비: Tweedie 모델과 평가 기준
from sklearn.linear_model import TweedieRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_validate

# [5·6단계 준비] 타깃을 "1년당 사고율"로, 가중치를 "가입 기간"으로 (offset 대신)
w_train = X_train["Exposure"]
y_train_rate = y_train / w_train


class RareLevelMergerRate(RareLevelMerger):
    """[5·6단계 준비] 1단계 RareLevelMerger는 y가 '사고 건수'라고 가정함
    여기서는 y가 '사고율'이라 사고율 × 가입 기간 = 사고 건수로 되돌려서 같은 기준(사고 200건)으로 판단"""

    def fit(self, X, y):
        return super().fit(X, np.asarray(y) * X["Exposure"].to_numpy())


def make_tweedie_model(power=1.0, alpha=0.0):
    # [5·6단계 준비] 1단계와 같은 전처리 (Exposure는 피처가 아니라서 넣지 않음 → 나머지 열은 자동으로 버려짐)
    encode = ColumnTransformer([
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ])
    # [5·6단계 준비] link="log": 예측 = exp(계수 합) → 사고율이 항상 0보다 큼 (포아송 회귀와 같은 모양)
    model = TweedieRegressor(power=power, alpha=alpha, link="log", max_iter=1000)
    return Pipeline([("rare", RareLevelMergerRate()), ("encode", encode), ("model", model)])


# ---------------------------------------------------------------- 5·6단계 준비: 평가 기준을 모든 후보에 똑같이
# power마다 Tweedie 자체 deviance는 기준이 달라서 서로 비교할 수 없음
# → 4단계와 같은 "사고 건수의 Poisson deviance"로 고정 (예측 사고율 × 가입 기간 = 예측 사고 건수)
# sklearn은 점수가 클수록 좋다고 보기 때문에 deviance·RMSE는 음수(-)로 돌려줌
def _counts(est, X, y):
    e = X["Exposure"]
    return y * e, est.predict(X) * e          # (실제 사고 건수, 예측 사고 건수)


def neg_poisson_dev(est, X, y):
    true, pred = _counts(est, X, y)
    return -mean_poisson_deviance(true, pred)


def neg_rmse(est, X, y):
    true, pred = _counts(est, X, y)
    return -np.sqrt(mean_squared_error(true, pred))


def r2(est, X, y):
    true, pred = _counts(est, X, y)
    return r2_score(true, pred)


SCORING = {"poisson_dev": neg_poisson_dev, "rmse": neg_rmse, "r2": r2}

# [5·6단계 준비] 교차 검증도 3단계처럼 "묶음 단위"로: 같은 계약이 학습 겹과 검증 겹에 동시에 들어가지 않게
cv = GroupKFold(n_splits=5)
print("[5·6 준비] 타깃 = 사고율, 가중치 = Exposure, 교차 검증 = GroupKFold 5겹")

[5·6 준비] 타깃 = 사고율, 가중치 = Exposure, 교차 검증 = GroupKFold 5겹


In [ ]:
# ================================================================ 5단계: 교차 검증 (포아송 = Tweedie power 1)
# [5단계] 슬라이드의 KFold를 묶음 버전(GroupKFold)으로 사용. groups로 묶음 번호를 넘김
# [5단계] params의 sample_weight는 겹마다 해당 행만 잘려서 모델에 전달됨
cv_pois = cross_validate(make_tweedie_model(power=1.0), X_train, y_train_rate,
                         groups=groups_train, cv=cv, scoring=SCORING,
                         params={"model__sample_weight": w_train})

print("[5] 포아송(power=1) 5겹 교차 검증")
for name, label in [("poisson_dev", "Poisson deviance"), ("rmse", "RMSE"), ("r2", "R²")]:
    s = cv_pois[f"test_{name}"]
    s = -s if name != "r2" else s                     # 음수로 받은 deviance·RMSE를 다시 양수로
    print(f"    {label:16s} 평균 {s.mean():.4f} (겹별 흔들림 ±{s.std():.4f})")

# ---------------------------------------------------------------- 5단계: 과대산포 확인
# [5단계] 산포 지수 = Pearson chi2 / 자유도. 1이면 포아송 가정이 맞고, 1보다 크면 과대산포
pois_full = make_tweedie_model(power=1.0).fit(X_train, y_train_rate, model__sample_weight=w_train)
mu = pois_full.predict(X_train) * w_train                      # 예측 사고 건수
n_params = pois_full.named_steps["model"].coef_.size + 1       # 계수 + 절편
dispersion = (((y_train - mu) ** 2) / mu).sum() / (len(y_train) - n_params)
print(f"    산포 지수 {dispersion:.2f} → 1보다 크면 과대산포 (6단계에서 Tweedie power로 보완)")

[5] 포아송(power=1) 5겹 교차 검증
    Poisson deviance 평균 0.3184 (겹별 흔들림 ±0.0038)
    RMSE             평균 0.2364 (겹별 흔들림 ±0.0014)
    R²               평균 0.0108 (겹별 흔들림 ±0.0019)
    산포 지수 2.56 → 1보다 크면 과대산포 (6단계에서 Tweedie power로 보완)


In [ ]:
# ================================================================ 6단계: GridSearchCV로 power · alpha 탐색
# [6단계] power: 분산이 평균에 따라 얼마나 커지는지 (1 = 포아송, 클수록 과대산포를 크게 반영)
# [6단계] alpha: L2(릿지) 규제 세기 (0 = 규제 없음). 행이 54만 개라 큰 효과는 없을 수 있음
param_grid = {
    "model__power": [1.0, 1.2, 1.4, 1.6, 1.8],
    "model__alpha": [0.0, 1e-4, 1e-3, 1e-2],
}
grid = GridSearchCV(make_tweedie_model(), param_grid, cv=cv, scoring=SCORING,
                    refit="poisson_dev",        # [6단계] 최종 선택 기준 = Poisson deviance (주 지표)
                    n_jobs=-1)
# [6단계] groups → GroupKFold로, model__sample_weight → 겹마다 잘려서 TweedieRegressor.fit으로 전달
grid.fit(X_train, y_train_rate, groups=groups_train, model__sample_weight=w_train)

# [6단계] 조합별 교차 검증 Poisson deviance 표 (작을수록 좋음)
res = pd.DataFrame(grid.cv_results_)
table = res.pivot(index="param_model__power", columns="param_model__alpha",
                  values="mean_test_poisson_dev").mul(-1)
print("[6] 조합별 교차 검증 Poisson deviance (행 = power, 열 = alpha)")
print(table.round(5).to_string())
print(f"\n    최적 조합: {grid.best_params_} → deviance {-grid.best_score_:.5f}")
print(f"    5단계 포아송(power=1, alpha=0): {-cv_pois['test_poisson_dev'].mean():.5f}")
# [6단계] 해석: 두 점수 차이가 5단계의 "겹별 흔들림(±)"보다 작으면 사실상 같은 성능
#         → 그때는 더 단순한 포아송(power=1)을 써도 되고, p값을 해석할 땐 과대산포를 반영한 쪽이 더 정직함
gap = grid.best_score_ - cv_pois["test_poisson_dev"].mean()     # 양수 = 포아송보다 deviance가 줄어든 만큼
print(f"    포아송 대비 개선 폭 {gap:.5f} vs 겹별 흔들림 ±{cv_pois['test_poisson_dev'].std():.4f}")

[6] 조합별 교차 검증 Poisson deviance (행 = power, 열 = alpha)
param_model__alpha   0.0000   0.0001   0.0010   0.0100
param_model__power                                    
1.0                 0.31839  0.31838  0.31840  0.31892
1.2                 0.31838  0.31837  0.31838  0.31870
1.4                 0.31840  0.31840  0.31840  0.31856
1.6                 0.31847  0.31846  0.31845  0.31848
1.8                 0.31856  0.31856  0.31853  0.31848

    최적 조합: {'model__alpha': 0.0001, 'model__power': 1.2} → deviance 0.31837
    5단계 포아송(power=1, alpha=0): 0.31839
    포아송 대비 개선 폭 0.00001 vs 겹별 흔들림 ±0.0038


In [ ]:
# ================================================================ 6단계 마무리: test는 여기서 딱 한 번
# [6단계 마무리] GridSearchCV는 최적 조합으로 train 전체를 다시 학습해 둠 (refit) → best_estimator_
best = grid.best_estimator_
pred_test_tw = best.predict(X_test) * X_test["Exposure"]      # 예측 사고율 × 가입 기간 = 예측 사고 건수

# [6단계 마무리] 4단계 포아송과 같은 지표(evaluate)로 비교 → 7단계 PPT 표로 사용
final = pd.DataFrame({
    "4단계 포아송(statsmodels)": evaluate(y_test, pred_test),
    f"6단계 Tweedie {grid.best_params_}": evaluate(y_test, pred_test_tw),
}).T
print("[6 마무리] test 비교")
print(final.round(4).to_string())
print(f"    예상 사고 건수 합: {pred_test_tw.sum():,.0f}건 / 실제 {y_test.sum():,}건")

[6 마무리] test 비교
                                                           Poisson deviance    RMSE      R²
4단계 포아송(statsmodels)                                                 0.3209  0.2379  0.0093
6단계 Tweedie {'model__alpha': 0.0001, 'model__power': 1.2}            0.3209  0.2379  0.0091
    예상 사고 건수 합: 7,225건 / 실제 7,246건
